# IAU Projekt
## Fáza 1: Prieskumná analýza dát

**Predmet:** Inteligentná analýza údajov (IAU)<br>
**Akademický rok / semester:** 2026/2027<br>
**Cvičiaci:** Ing. Laura Krajčovičová<br>
**Číslo datasetu:** 038<br>
**Autori:** Adam Kavoň, Jakub Žúbor

## Úvod

Cieľom projektu je predikovať zlyhanie (premenná `failure`) jednotky na výrobu vzduchu vo vlakoch na základe senzorových dát.

Dáta sa skladajú z troch súborov:
- `gateways.csv` – informácie o bránach (gateway), ktoré zbierajú dáta (výrobca, firmvér, model, poloha, dátum inštalácie, ...)
- `readings.csv` – senzorové merania
- `units.csv` – informácie o jednotkách (APU)

V tejto fáze vykonávame prieskumnú analýzu, identifikujeme problémy v dátach a štatisticky overujeme dve hypotézy.

## Riešenie

Na začiatok je potrebné si importovať všetky knižnice, ktoré budeme pri práci s dátami potrebovať.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

## 1.1 A) Analýza štruktúr dát

Ako prvé si vypíšeme obsah dátových súborov, aby sme zistili ich štruktúru 

In [29]:
for name in ["units", "readings", "gateways"]:
    print(f"===== {name} =====")
    with open(f"data/{name}.csv", "r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f):
            print(i, repr(line))
            if i >= 4:
                break
    print()

===== units =====
0 'install_region,id_unit,owner_operator,last_overhaul,modelCode,dutyClass,depotId,hoursService,numberSerial\n'
1 'New Shane,1,Antunes,"10/03/2023, 00:00:00",APU-12,3,106,34094.60000,SG-1619-42643\n'
2 'Tomar,2,Wiek GmbH & Co. KG,"03/26/2020, 00:00:00",XC-7,1,111,,ZD-8362-37250\n'
3 'Gouveia,3,Wagner,"06/10/2019, 00:00:00",APU-15,normal,79,27642.50000,IG-0808-58786\n'
4 'Vila Nova de Foz Côa,4,Lima,2015-10-14,APU-12,1,104,3461.60000,OQ-3972-90589\n'

===== readings =====
0 'id_unit\tts_cycle\tmotor_current\tfilter_dp\toilTemperature\tpressure_reservoir\tpressureOil\trms_vibration\tflowmeter\thumidityIntake\tambientNoise\tdewpoint_discharge\taux_current\tFailure\n'
1 'U00890\t01/05/2023, 02:01:00\t6.03488\t12.35880\t52.66939\t8.61464\t3.95319\t4.28315\t31.04074\t86.68502\t64.70465\t7.37748\t2.07710\t0\n'
2 'U90104\t2023/01/07\t5.88784\t20.12438\t77.76356\t8.41482\t3.15135\t2.52806\t29.53863\t88.80272\t84.87385\t11.06149\t3.47304\t0\n'
3 'U01151\t28 Jan 2023\t0.61381\t3

In [24]:
units_data = pd.read_csv("data/units.csv")
units_data

,install_region,id_unit,owner_operator,last_overhaul,modelCode,dutyClass,depotId,hoursService,numberSerial
0,New Shane,1,Antunes,"10/03/2023, 00:00:00",APU-12,3,106,34094.6,SG-1619-42643
1,Tomar,2,Wiek GmbH & Co. KG,"03/26/2020, 00:00:00",XC-7,1,111,NaN,ZD-8362-37250
2,Gouveia,3,Wagner,"06/10/2019, 00:00:00",APU-15,normal,79,27642.5,IG-0808-58786
3,Vila Nova de Foz Côa,4,Lima,2015-10-14,APU-12,1,104,3461.6,OQ-3972-90589
4,Lippstadt,5,Gaspar Ferreira S.A.,"04/26/2023, 00:00:00",APU-12,1,18,4657.5,OL-0793-38153
...,...,...,...,...,...,...,...,...,...
1995,Winterston,1996,Leal S/A,18 Dec 2023,XC-7,3,33,NaN,HP-7622-05671
1996,Goslar,1997,Johnson-Sanchez,08 Feb 2017,APU-20,L,7,14979.5,FZ-9328-81090
1997,Vila Nova de Santo André,1998,Antunes,2016/05/03,APU-12,2,68,9928.8,BV-6559-48073
1998,Lippstadt,1999,Antunes,2024/09/20,APU-20,2,8,29535.8,NJ-5380-69337


In [28]:
readings_data = pd.read_csv("data/readings.csv", sep="\t")
readings_data

,id_unit,ts_cycle,motor_current,filter_dp,oilTemperature,pressure_reservoir,pressureOil,rms_vibration,flowmeter,humidityIntake,ambientNoise,dewpoint_discharge,aux_current,Failure
0,U00890,"01/05/2023, 02:01:00",6.03488,12.35880,52.66939,8.61464,3.95319,4.28315,31.04074,86.68502,64.70465,7.37748,2.07710,0
1,U90104,2023/01/07,5.88784,20.12438,77.76356,8.41482,3.15135,2.52806,29.53863,88.80272,84.87385,11.06149,3.47304,0
2,U01151,28 Jan 2023,0.61381,34.03841,41.90010,7.82918,2.98944,NaN,26.94321,40.04785,84.85633,7.64956,2.75321,1
3,U00870,02 Dec 2023,7.36066,30.62395,64.68951,8.79319,3.51568,1.12909,25.36901,78.50330,61.72889,8.17558,2.15853,0
4,U01371,2023-07-13,4.25880,40.91751,62.48404,8.82556,3.55145,4.25855,34.66068,33.28814,70.13243,5.51669,1.47229,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13934,U01374,06 Nov 2023,7.71051,NaN,64.36301,8.83308,3.95782,2.89603,28.43260,64.50666,78.69199,11.35168,1.73991,1
13935,U01891,2023-04-14,5.22230,12.13601,60.34316,9.78535,4.58946,2.27918,23.82831,29.91265,81.72715,11.50342,3.82224,0
13936,U00293,"03/11/2023, 19:52:00",3.40618,13.44212,61.52971,8.60193,3.01348,2.19284,26.14997,91.37227,76.51522,7.94773,4.15677,0
13937,U00167,2023-08-27,5.66443,42.84716,50.37179,8.02551,2.67634,5.28309,25.52513,66.75342,79.44340,7.43455,3.37710,1


In [26]:
gateways_data = pd.read_csv("data/gateways.csv", sep=";")
gateways_data

,id_unit,macAddress,Ipv4,id_gateway,signal_dbm,dateInstall,lat,vendor,version_firmware,hardwareModel,depotId,technician,lon
0,1905.0,78:c7:eb:c0:ea:9b,145.254.205.111,GW-00000,-72.6,18 Jul 2018,40.69655,Johann GmbH,v2.7.0,GWX-200,97,Gonçalo do Lopes,-7.35887
1,467.0,ed:67:e9:95:b0:4c,85.40.9.139,GW-00001,-82.7,2020/11/16,39.00417,Wagner,v1.2.5,GWX-300,109,Evelyn Tintzmann,-7.52314
2,1884.0,14:82:e1:29:a9:85,25.95.112.105,GW-00002,-61.2,2019-09-14,38.66537,Metz GmbH & Co. KGaA,v3.4.1,GWX-300,35,Ângela Oliveira,-6.27268
3,325.0,6a:a1:1e:b7:67:0f,38.89.178.61,GW-00003,-78.1,19 Sep 2020,41.46158,Riehl Zahn GmbH & Co. OHG,v2.7.0,GWX-100,65,Audrey Levine,-8.73627
4,765.0,fc:c5:29:5f:f6:92,228.109.20.169,GW-00004,-58.4,"02/18/2023, 00:00:00",41.13818,Wähner,v3.4.1,GWX-100,4,Dr. Rosmarie Beier,-6.68186
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15995,NaN,fe:41:b4:d4:28:de,185.89.83.228,GW-15995,-64.7,2017/02/01,41.66279,Cameron-Smith,v2.7.0,NODE-9,5,Tiago Miranda,-7.05795
15996,NaN,4f:57:17:1d:57:e9,111.44.166.49,GW-15996,-59.9,2016/12/08,39.70297,"Pierce, Fuller and Lester",v1.2.5,GWX-300,73,Erika Cox,-7.37284
15997,NaN,30:e8:13:74:e9:51,239.119.60.122,GW-15997,-73.9,25 Nov 2022,38.78948,Lozano-Nguyen,v2.7.0,NODE-9,97,Catarina Lima,-7.70229
15998,NaN,6d:37:25:3e:c9:6c,127.190.201.48,GW-15998,-80.1,2021-03-28,38.76347,Möchlichen,v3.4.1,GWX-300,40,Gonçalo do Lopes,-7.04316


In [15]:
with open("data/gateways.csv", "r", encoding="utf-8", errors="replace") as f:
    for i, line in enumerate(f):
        print(i, repr(line))
        if i >= 15:
            break

0 'id_unit;macAddress;Ipv4;id_gateway;signal_dbm;dateInstall;lat;vendor;version_firmware;hardwareModel;depotId;technician;lon\n'
1 '1905;78:c7:eb:c0:ea:9b;145.254.205.111;GW-00000;-72.60000;18 Jul 2018;40.69655;Johann GmbH;v2.7.0;GWX-200;97;Gonçalo do Lopes;-7.35887\n'
2 '467;ed:67:e9:95:b0:4c;85.40.9.139;GW-00001;-82.70000;2020/11/16;39.00417;Wagner;v1.2.5;GWX-300;109;Evelyn Tintzmann;-7.52314\n'
3 '1884;14:82:e1:29:a9:85;25.95.112.105;GW-00002;-61.20000;2019-09-14;38.66537;Metz GmbH & Co. KGaA;v3.4.1;GWX-300;35;Ângela Oliveira;-6.27268\n'
4 '325;6a:a1:1e:b7:67:0f;38.89.178.61;GW-00003;-78.10000;19 Sep 2020;41.46158;Riehl Zahn GmbH & Co. OHG;v2.7.0;GWX-100;65;Audrey Levine;-8.73627\n'
5 '765;fc:c5:29:5f:f6:92;228.109.20.169;GW-00004;-58.40000;02/18/2023, 00:00:00;41.13818;Wähner;v3.4.1;GWX-100;4;Dr. Rosmarie Beier;-6.68186\n'
6 '885;59:89:a2:1f:89:60;138.190.112.50;GW-00005;-69.70000;2024/06/17;40.17985;Wirth Mende GmbH & Co. KG;v1.2.5;GWX-100;6;Jozef Henschel B.Sc.;-8.30899\n'
7 '24;